# Práctica 10 — Correlación y regresión lineal

**Curso:** Estadística Aplicada con Python · **Clase 10** · **Tiempo estimado:** 75-100 minutos
**Notebook de clase:** `notebooks_clase/clase10_correlacion_regresion_lineal.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios con simulación usan tolerancias basadas en el error estándar, así que un resultado correcto no debería fallar por azar.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Los ejemplos pequeños y las simulaciones son *ejemplos ilustrativos* sintéticos; los ejercicios 3 a 5 usan los datos de California housing (CSV local, descarga o alternativa sintética, como en la clase).

---
## Celda 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

In [ ]:
def generar_california_sintetico(n=5000, seed=42):
    """Genera un dataset SINTÉTICO con las mismas columnas que California housing (alternativa si no hay CSV ni descarga).
    Las relaciones están inventadas: sirve para practicar, no para sacar conclusiones sobre California."""
    g = np.random.default_rng(seed)
    MedInc = np.clip(g.lognormal(1.25, 0.5, n), 0.5, 15)
    HouseAge = g.integers(1, 53, n).astype(float)
    AveRooms = np.clip(3.0 + 0.35 * MedInc + g.normal(0, 0.7, n), 2, 9)
    AveBedrms = np.clip(0.2 * AveRooms + g.normal(0, 0.05, n), 0.6, 2)
    Population = np.round(g.lognormal(7.0, 0.6, n))
    AveOccup = np.clip(g.lognormal(1.05, 0.25, n), 1.2, 5.5)
    Latitude = np.round(g.uniform(32.5, 42.0, n), 2)
    Longitude = np.round(g.uniform(-124.3, -114.5, n), 2)
    valor = (0.5 + 0.42 * MedInc + 0.012 * HouseAge - 0.15 * AveRooms - 0.3 * AveOccup
             + 0.05 * (37 - Latitude) + g.normal(0, 0.6, n))
    MedHouseVal = np.clip(valor, 0.15, 5.00001)          # censura en 5,00001, como el original
    return pd.DataFrame({"MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms, "AveBedrms": AveBedrms,
                         "Population": Population, "AveOccup": AveOccup, "Latitude": Latitude,
                         "Longitude": Longitude, "MedHouseVal": MedHouseVal})


def cargar_california():
    """Devuelve (DataFrame, origen): CSV local, si no descarga de scikit-learn, si no datos sintéticos."""
    for ruta in ("../datos/california_housing.csv", "datos/california_housing.csv", "california_housing.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV local"
    try:
        from sklearn.datasets import fetch_california_housing
        datos = fetch_california_housing(as_frame=True).frame
        return datos.sample(n=5000, random_state=42).reset_index(drop=True), "scikit-learn"
    except Exception:
        return generar_california_sintetico(), "sintético"


df_bruto, origen = cargar_california()
print("Origen de los datos:", origen, "|", df_bruto.shape[0], "filas")

In [ ]:
# Preparación: quitamos distritos con valores medios absurdos y creamos la variable de zona
df = df_bruto[(df_bruto["AveRooms"] < 10) & (df_bruto["AveOccup"] < 6)].copy()
df["zona"] = np.where(df["Latitude"] >= 36, "Norte", "Sur")
print(f"Distritos usados: {len(df)} de {len(df_bruto)}")

In [ ]:
def r_de(x, y):
    """Correlación de Pearson entre dos vectores."""
    return np.corrcoef(x, y)[0, 1]

## Ejercicio 1 — Pearson y Spearman a mano

Siete puntos con una relación **creciente pero curva** (*ejemplo ilustrativo*): `y = x²`. Calcula, sin usar `np.corrcoef` ni `stats.spearmanr`:

- `cov_xy`: covarianza muestral (`ddof=1`),
- `r_pearson`: covarianza dividida entre el producto de las desviaciones típicas (con `ddof=1`),
- `rho_spearman`: correlación de Pearson calculada sobre los **rangos** (`stats.rankdata`).

Pregunta para pensar: ¿por qué `rho_spearman` es exactamente 1 y `r_pearson` no?

In [ ]:
x1 = np.arange(1, 8, dtype=float)
y1 = x1 ** 2

cov_xy = None          # TODO: suma de productos de las desviaciones / (n - 1)
r_pearson = None       # TODO
rho_spearman = None    # TODO: Pearson de los rangos de x1 y de y1

# Comprobación automática
assert all(v is not None for v in (cov_xy, r_pearson, rho_spearman)), "Completa las líneas marcadas con TODO"
assert np.isclose(cov_xy, np.cov(x1, y1)[0, 1]), "la covarianza usa ddof=1: divide entre n - 1"
assert np.isclose(r_pearson, np.corrcoef(x1, y1)[0, 1]), "r = cov / (sx · sy), con ddof=1 en las tres cantidades"
assert np.isclose(rho_spearman, 1.0), "los rangos de x1 y de y1 coinciden: la correlación de los rangos vale 1"
assert 0.95 < r_pearson < 1.0, "Pearson es alto pero menor que 1: la relación es curva"
print(f"Covarianza = {cov_xy:.2f}   Pearson = {r_pearson:.4f}   Spearman = {rho_spearman:.1f}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Mínimos cuadrados a mano

Cinco pares (*ejemplo ilustrativo*): `x = 2, 4, 6, 8, 10` (inversión en publicidad, miles de €) e `y = 3, 7, 8, 12, 15` (ventas, miles de €). Calcula con las fórmulas (sin `polyfit` ni `statsmodels`):

- `b1` (pendiente) y `b0` (intercepto),
- `pred_7`: predicción para `x = 7`,
- `ssr`: suma de cuadrados de los residuos, y `r2 = 1 - ssr / sst`, con `sst` la suma de cuadrados total de `y` respecto a su media.

In [ ]:
x2 = np.array([2, 4, 6, 8, 10], dtype=float)
y2 = np.array([3, 7, 8, 12, 15], dtype=float)

b1 = None        # TODO: Sxy / Sxx
b0 = None        # TODO: media de y menos b1 por la media de x
pred_7 = None    # TODO
ssr = None       # TODO: suma de (y - predicho)²
sst = None       # TODO: suma de (y - media de y)²
r2 = None        # TODO

# Comprobación automática
assert all(v is not None for v in (b1, b0, pred_7, ssr, sst, r2)), "Completa las líneas marcadas con TODO"
assert np.isclose(b1, 1.45), "b1 = Sxy / Sxx = 58 / 40"
assert np.isclose(b0, 0.3), "b0 = 9 - 1,45 · 6"
assert np.isclose(pred_7, 10.45), "pred_7 = b0 + b1 · 7"
assert np.isclose(ssr, 1.9) and np.isclose(sst, 86), "revisa los residuos y la suma total"
assert np.isclose(r2, np.corrcoef(x2, y2)[0, 1] ** 2), "con una sola variable, R² = r²"
print(f"b1 = {b1:.2f}   b0 = {b0:.2f}   predicción en x=7 = {pred_7:.2f}   R² = {r2:.4f}")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Regresión múltiple con `statsmodels`

Con los datos de California (`df`, ya preparado en el setup), ajusta `MedHouseVal ~ MedInc + HouseAge` con `smf.ols(...).fit()` y extrae:

- `modelo`: el resultado del ajuste,
- `coef_ingreso`: coeficiente de `MedInc`, e `ic_ingreso`: su intervalo de confianza al 95 % como tupla `(inferior, superior)`,
- `p_ingreso`: su p-value,
- `efecto_10_anios`: cambio esperado del valor por 10 años más de edad de las viviendas, **a igualdad de renta**,
- `r2` y `r2_aj`: R² y R² corregido.

In [ ]:
modelo = None            # TODO: smf.ols("MedHouseVal ~ MedInc + HouseAge", data=df).fit()
coef_ingreso = None      # TODO
ic_ingreso = None        # TODO: tupla (inferior, superior) a partir de conf_int()
p_ingreso = None         # TODO
efecto_10_anios = None   # TODO
r2 = None                # TODO
r2_aj = None             # TODO

# Comprobación automática (contra un cálculo independiente con álgebra lineal)
assert all(v is not None for v in (modelo, coef_ingreso, ic_ingreso, p_ingreso, efecto_10_anios, r2, r2_aj)), "Completa las líneas marcadas con TODO"
X_ref = np.column_stack([np.ones(len(df)), df["MedInc"], df["HouseAge"]])
y_ref = df["MedHouseVal"].values
beta = np.linalg.lstsq(X_ref, y_ref, rcond=None)[0]
res_ref = y_ref - X_ref @ beta
gl_ref = len(df) - 3
s2 = (res_ref ** 2).sum() / gl_ref
ee_ref = np.sqrt(np.diag(s2 * np.linalg.inv(X_ref.T @ X_ref)))
t_ref = stats.t.ppf(0.975, gl_ref)
assert np.isclose(coef_ingreso, beta[1]), "el coeficiente de MedInc está en modelo.params"
assert np.allclose(ic_ingreso, (beta[1] - t_ref * ee_ref[1], beta[1] + t_ref * ee_ref[1]), atol=1e-6), "usa modelo.conf_int() y toma la fila de MedInc"
assert ic_ingreso[0] < coef_ingreso < ic_ingreso[1], "el coeficiente debe caer dentro de su intervalo"
assert 0 <= p_ingreso < 0.05, "revisa modelo.pvalues"
assert np.isclose(efecto_10_anios, 10 * beta[2]), "el efecto de 10 años es 10 veces el coeficiente de HouseAge"
assert np.isclose(r2, 1 - (res_ref ** 2).sum() / ((y_ref - y_ref.mean()) ** 2).sum()), "R² = 1 - SSR/SST; está en modelo.rsquared"
assert r2_aj < r2, "el R² corregido siempre es menor que R² (con más de una variable)"
print(f"MedInc: {coef_ingreso:.4f}   IC 95 %: ({ic_ingreso[0]:.4f}, {ic_ingreso[1]:.4f})   R² = {r2:.4f}   R² corregido = {r2_aj:.4f}")
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Previsión con dos intervalos

Con el modelo simple `m_ej = smf.ols("MedHouseVal ~ MedInc", data=df).fit()`, completa:

- `intervalos(modelo, medinc)`: devuelve un diccionario con `"pred"` (valor predicho), `"ic_media"` (tupla del IC 95 % de la media) e `"int_pred"` (tupla del intervalo de predicción 95 %). Pista: `modelo.get_prediction(...).summary_frame(alpha=0.05)`.
- `en_rango(medinc)`: `True` si `medinc` está dentro del rango observado de `df["MedInc"]`.

In [ ]:
m_ej = smf.ols("MedHouseVal ~ MedInc", data=df).fit()

def intervalos(modelo, medinc):
    """Predicción, IC 95 % de la media e intervalo de predicción 95 % para un valor de MedInc."""
    # TODO: usa get_prediction sobre un DataFrame con la columna MedInc
    return None

def en_rango(medinc):
    """True si medinc está dentro del rango observado de MedInc."""
    # TODO
    return None

# Comprobación automática
r4 = intervalos(m_ej, 4.0)
assert r4 is not None and en_rango(4.0) is not None, "Completa las funciones (TODO)"
b0_, b1_ = m_ej.params["Intercept"], m_ej.params["MedInc"]
assert np.isclose(r4["pred"], b0_ + b1_ * 4.0), "la predicción es b0 + b1 · x"
assert r4["ic_media"][0] < r4["pred"] < r4["ic_media"][1], "la predicción debe estar dentro del IC de la media"
assert r4["int_pred"][0] < r4["ic_media"][0] and r4["int_pred"][1] > r4["ic_media"][1], "el intervalo de predicción contiene al IC de la media y es más ancho"
ancho_media = r4["ic_media"][1] - r4["ic_media"][0]
ancho_pred = r4["int_pred"][1] - r4["int_pred"][0]
assert ancho_pred > 10 * ancho_media, "con miles de datos el IC de la media es mucho más estrecho que el de predicción"
assert en_rango(4.0) is True and en_rango(30.0) is False and en_rango(0.0) is False, "compara con el mínimo y el máximo de df['MedInc']"
print(f"Anchos: IC de la media = {ancho_media:.3f}   intervalo de predicción = {ancho_pred:.3f}")
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Diagnóstico de residuos

Con `m_ej` (ejercicio 4), calcula:

- `techo`: máscara booleana de los distritos con `MedHouseVal >= 5` (valores censurados), `n_techo`: cuántos son,
- `res_techo` y `res_resto`: residuo medio de los censurados y del resto,
- `cook`: distancias de Cook (`m_ej.get_influence().cooks_distance[0]`) y `pos_cook_max`: posición de la mayor,
- `corr_res_ajust`: correlación entre residuos y valores ajustados.

Pregunta para pensar: ¿por qué los residuos de los distritos censurados son en promedio **positivos**?

In [ ]:
residuos = m_ej.resid.values
ajustados = m_ej.fittedvalues.values

techo = None            # TODO: máscara booleana (df["MedHouseVal"] >= 5).values
n_techo = None          # TODO
res_techo = None        # TODO
res_resto = None        # TODO
cook = None             # TODO
pos_cook_max = None     # TODO
corr_res_ajust = None   # TODO

# Comprobación automática
assert all(v is not None for v in (techo, n_techo, res_techo, res_resto, cook, pos_cook_max, corr_res_ajust)), "Completa las líneas marcadas con TODO"
assert n_techo == int((df["MedHouseVal"] >= 5).sum()) and n_techo > 0, "cuenta los distritos con MedHouseVal >= 5"
assert np.isclose(res_techo, residuos[techo].mean()) and np.isclose(res_resto, residuos[~techo].mean()), "usa la máscara techo y su negación"
assert res_techo > 0 > res_resto, "los censurados tienen residuo positivo: su valor real (sin recortar) sería aún mayor"
assert len(cook) == len(df) and pos_cook_max == int(np.argmax(cook)), "pos_cook_max es la posición del máximo de cook"
assert abs(corr_res_ajust) < 1e-8, "en OLS con intercepto, los residuos no se correlacionan con los valores ajustados"
print(f"Censurados: {n_techo}   residuo medio censurados / resto: {res_techo:+.2f} / {res_resto:+.2f}   Cook máx.: {cook.max():.3f}")
print("Ejercicio 5 correcto ✔")

---
## Reto final — La confusión que engaña

Una simulación (*ejemplo ilustrativo*): la variable `z` influye en `x` y en `y`, y `x` tiene un efecto **verdadero** de 0,3 sobre `y`. Un análisis ingenuo que ignore `z` sobrestima ese efecto.

Completa:

- `coef_crudo`, `ee_crudo`: coeficiente y error estándar de `x` en `y ~ x`,
- `coef_ajustado`, `ee_ajustado`: lo mismo en `y ~ x + z`,
- `sesgo`: `coef_crudo - coef_ajustado`.

Después escribe (en la celda de conclusiones) qué haría falta para atribuir a `x` un efecto **causal**.

In [ ]:
g = np.random.default_rng(11)
n_r = 3000
z_r = g.normal(0, 1, n_r)
x_r = 0.8 * z_r + g.normal(0, 1, n_r)
y_r = 0.3 * x_r + 1.0 * z_r + g.normal(0, 1, n_r)
datos_r = pd.DataFrame({"x": x_r, "y": y_r, "z": z_r})

coef_crudo = None       # TODO
ee_crudo = None         # TODO
coef_ajustado = None    # TODO
ee_ajustado = None      # TODO
sesgo = None            # TODO

# Comprobación automática (tolerancias basadas en el error estándar)
assert all(v is not None for v in (coef_crudo, ee_crudo, coef_ajustado, ee_ajustado, sesgo)), "Completa las líneas marcadas con TODO"
assert abs(coef_ajustado - 0.3) < 4 * ee_ajustado, "el modelo con z debe recuperar el efecto verdadero 0,3 (a menos de 4 errores estándar)"
assert coef_crudo - 0.3 > 4 * ee_crudo, "el modelo sin z sobrestima el efecto (más de 4 errores estándar)"
assert np.isclose(sesgo, coef_crudo - coef_ajustado) and sesgo > 0.3, "el sesgo es crudo - ajustado y aquí supera 0,3"
print(f"Efecto verdadero: 0,3   crudo: {coef_crudo:.3f} ± {ee_crudo:.3f}   ajustado: {coef_ajustado:.3f} ± {ee_ajustado:.3f}")
print("sesgo:", round(sesgo, 3))
print("Reto final correcto ✔")

**Tus conclusiones (escribe 3-4 frases):** ¿qué habría que hacer para poder decir que `x` *causa* `y`? ¿Cuándo es útil un modelo de regresión aunque no permita hablar de causalidad? ¿Qué límites de los datos de California (censura en 5,00001, datos de 1990, distritos y no viviendas individuales) deberías mencionar en un informe?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
x1 = np.arange(1, 8, dtype=float)
y1 = x1 ** 2

cov_xy = ((x1 - x1.mean()) * (y1 - y1.mean())).sum() / (len(x1) - 1)
r_pearson = cov_xy / (x1.std(ddof=1) * y1.std(ddof=1))
rho_spearman = np.corrcoef(stats.rankdata(x1), stats.rankdata(y1))[0, 1]

assert all(v is not None for v in (cov_xy, r_pearson, rho_spearman)), "Completa las líneas marcadas con TODO"
assert np.isclose(cov_xy, np.cov(x1, y1)[0, 1]), "la covarianza usa ddof=1: divide entre n - 1"
assert np.isclose(r_pearson, np.corrcoef(x1, y1)[0, 1]), "r = cov / (sx · sy), con ddof=1 en las tres cantidades"
assert np.isclose(rho_spearman, 1.0), "los rangos de x1 y de y1 coinciden: la correlación de los rangos vale 1"
assert 0.95 < r_pearson < 1.0, "Pearson es alto pero menor que 1: la relación es curva"
print(f"Covarianza = {cov_xy:.2f}   Pearson = {r_pearson:.4f}   Spearman = {rho_spearman:.1f}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
x2 = np.array([2, 4, 6, 8, 10], dtype=float)
y2 = np.array([3, 7, 8, 12, 15], dtype=float)

b1 = ((x2 - x2.mean()) * (y2 - y2.mean())).sum() / ((x2 - x2.mean()) ** 2).sum()
b0 = y2.mean() - b1 * x2.mean()
pred_7 = b0 + b1 * 7
ssr = ((y2 - (b0 + b1 * x2)) ** 2).sum()
sst = ((y2 - y2.mean()) ** 2).sum()
r2 = 1 - ssr / sst

assert all(v is not None for v in (b1, b0, pred_7, ssr, sst, r2)), "Completa las líneas marcadas con TODO"
assert np.isclose(b1, 1.45), "b1 = Sxy / Sxx = 58 / 40"
assert np.isclose(b0, 0.3), "b0 = 9 - 1,45 · 6"
assert np.isclose(pred_7, 10.45), "pred_7 = b0 + b1 · 7"
assert np.isclose(ssr, 1.9) and np.isclose(sst, 86), "revisa los residuos y la suma total"
assert np.isclose(r2, np.corrcoef(x2, y2)[0, 1] ** 2), "con una sola variable, R² = r²"
print(f"b1 = {b1:.2f}   b0 = {b0:.2f}   predicción en x=7 = {pred_7:.2f}   R² = {r2:.4f}")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
modelo = smf.ols("MedHouseVal ~ MedInc + HouseAge", data=df).fit()
coef_ingreso = modelo.params["MedInc"]
ic_ingreso = tuple(modelo.conf_int().loc["MedInc"])
p_ingreso = modelo.pvalues["MedInc"]
efecto_10_anios = 10 * modelo.params["HouseAge"]
r2 = modelo.rsquared
r2_aj = modelo.rsquared_adj

assert all(v is not None for v in (modelo, coef_ingreso, ic_ingreso, p_ingreso, efecto_10_anios, r2, r2_aj)), "Completa las líneas marcadas con TODO"
X_ref = np.column_stack([np.ones(len(df)), df["MedInc"], df["HouseAge"]])
y_ref = df["MedHouseVal"].values
beta = np.linalg.lstsq(X_ref, y_ref, rcond=None)[0]
res_ref = y_ref - X_ref @ beta
gl_ref = len(df) - 3
s2 = (res_ref ** 2).sum() / gl_ref
ee_ref = np.sqrt(np.diag(s2 * np.linalg.inv(X_ref.T @ X_ref)))
t_ref = stats.t.ppf(0.975, gl_ref)
assert np.isclose(coef_ingreso, beta[1]), "el coeficiente de MedInc está en modelo.params"
assert np.allclose(ic_ingreso, (beta[1] - t_ref * ee_ref[1], beta[1] + t_ref * ee_ref[1]), atol=1e-6), "usa modelo.conf_int() y toma la fila de MedInc"
assert ic_ingreso[0] < coef_ingreso < ic_ingreso[1], "el coeficiente debe caer dentro de su intervalo"
assert 0 <= p_ingreso < 0.05, "revisa modelo.pvalues"
assert np.isclose(efecto_10_anios, 10 * beta[2]), "el efecto de 10 años es 10 veces el coeficiente de HouseAge"
assert np.isclose(r2, 1 - (res_ref ** 2).sum() / ((y_ref - y_ref.mean()) ** 2).sum()), "R² = 1 - SSR/SST; está en modelo.rsquared"
assert r2_aj < r2, "el R² corregido siempre es menor que R² (con más de una variable)"
print(f"MedInc: {coef_ingreso:.4f}   IC 95 %: ({ic_ingreso[0]:.4f}, {ic_ingreso[1]:.4f})   R² = {r2:.4f}   R² corregido = {r2_aj:.4f}")
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
m_ej = smf.ols("MedHouseVal ~ MedInc", data=df).fit()

def intervalos(modelo, medinc):
    """Predicción, IC 95 % de la media e intervalo de predicción 95 % para un valor de MedInc."""
    fila = modelo.get_prediction(pd.DataFrame({"MedInc": [medinc]})).summary_frame(alpha=0.05).iloc[0]
    return {"pred": fila["mean"],
            "ic_media": (fila["mean_ci_lower"], fila["mean_ci_upper"]),
            "int_pred": (fila["obs_ci_lower"], fila["obs_ci_upper"])}

def en_rango(medinc):
    """True si medinc está dentro del rango observado de MedInc."""
    return bool(df["MedInc"].min() <= medinc <= df["MedInc"].max())

r4 = intervalos(m_ej, 4.0)
assert r4 is not None and en_rango(4.0) is not None, "Completa las funciones (TODO)"
b0_, b1_ = m_ej.params["Intercept"], m_ej.params["MedInc"]
assert np.isclose(r4["pred"], b0_ + b1_ * 4.0), "la predicción es b0 + b1 · x"
assert r4["ic_media"][0] < r4["pred"] < r4["ic_media"][1], "la predicción debe estar dentro del IC de la media"
assert r4["int_pred"][0] < r4["ic_media"][0] and r4["int_pred"][1] > r4["ic_media"][1], "el intervalo de predicción contiene al IC de la media y es más ancho"
ancho_media = r4["ic_media"][1] - r4["ic_media"][0]
ancho_pred = r4["int_pred"][1] - r4["int_pred"][0]
assert ancho_pred > 10 * ancho_media, "con miles de datos el IC de la media es mucho más estrecho que el de predicción"
assert en_rango(4.0) is True and en_rango(30.0) is False and en_rango(0.0) is False, "compara con el mínimo y el máximo de df['MedInc']"
print(f"Anchos: IC de la media = {ancho_media:.3f}   intervalo de predicción = {ancho_pred:.3f}")
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
residuos = m_ej.resid.values
ajustados = m_ej.fittedvalues.values

techo = (df["MedHouseVal"] >= 5).values
n_techo = int(techo.sum())
res_techo = residuos[techo].mean()
res_resto = residuos[~techo].mean()
cook = m_ej.get_influence().cooks_distance[0]
pos_cook_max = int(np.argmax(cook))
corr_res_ajust = np.corrcoef(residuos, ajustados)[0, 1]

assert all(v is not None for v in (techo, n_techo, res_techo, res_resto, cook, pos_cook_max, corr_res_ajust)), "Completa las líneas marcadas con TODO"
assert n_techo == int((df["MedHouseVal"] >= 5).sum()) and n_techo > 0, "cuenta los distritos con MedHouseVal >= 5"
assert np.isclose(res_techo, residuos[techo].mean()) and np.isclose(res_resto, residuos[~techo].mean()), "usa la máscara techo y su negación"
assert res_techo > 0 > res_resto, "los censurados tienen residuo positivo: su valor real (sin recortar) sería aún mayor"
assert len(cook) == len(df) and pos_cook_max == int(np.argmax(cook)), "pos_cook_max es la posición del máximo de cook"
assert abs(corr_res_ajust) < 1e-8, "en OLS con intercepto, los residuos no se correlacionan con los valores ajustados"
print(f"Censurados: {n_techo}   residuo medio censurados / resto: {res_techo:+.2f} / {res_resto:+.2f}   Cook máx.: {cook.max():.3f}")
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
g = np.random.default_rng(11)
n_r = 3000
z_r = g.normal(0, 1, n_r)
x_r = 0.8 * z_r + g.normal(0, 1, n_r)
y_r = 0.3 * x_r + 1.0 * z_r + g.normal(0, 1, n_r)
datos_r = pd.DataFrame({"x": x_r, "y": y_r, "z": z_r})

m_c = smf.ols("y ~ x", data=datos_r).fit()
m_a = smf.ols("y ~ x + z", data=datos_r).fit()
coef_crudo, ee_crudo = m_c.params["x"], m_c.bse["x"]
coef_ajustado, ee_ajustado = m_a.params["x"], m_a.bse["x"]
sesgo = coef_crudo - coef_ajustado

assert all(v is not None for v in (coef_crudo, ee_crudo, coef_ajustado, ee_ajustado, sesgo)), "Completa las líneas marcadas con TODO"
assert abs(coef_ajustado - 0.3) < 4 * ee_ajustado, "el modelo con z debe recuperar el efecto verdadero 0,3 (a menos de 4 errores estándar)"
assert coef_crudo - 0.3 > 4 * ee_crudo, "el modelo sin z sobrestima el efecto (más de 4 errores estándar)"
assert np.isclose(sesgo, coef_crudo - coef_ajustado) and sesgo > 0.3, "el sesgo es crudo - ajustado y aquí supera 0,3"
print(f"Efecto verdadero: 0,3   crudo: {coef_crudo:.3f} ± {ee_crudo:.3f}   ajustado: {coef_ajustado:.3f} ± {ee_ajustado:.3f}")
print("sesgo:", round(sesgo, 3))
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (correlación vs. causalidad, lectura del output, R² corregido, intervalos de previsión, diagnóstico de residuos)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*